# NutriCare Food Recognition Model

### A reproducible walkthrough of the model NutriCare actually uses

**Purpose:** present the real food-image classifier, its dataset lineage, preparation, training evidence, held-out evaluation, and the same inference path used by the application.

> **Model selection is deliberate:** NutriCare's primary scanner model is the **80-class Indian-food MobileNetV3 Small** checkpoint at `models/v2/food_classifier_indian_80_224.pt`. The 12-class checkpoint is a technical fallback only; it is not the normal scanner model. This notebook neither retrains nor modifies either model.

**Evidence standard.** Numbers below are read from committed audit/metrics files. The repository's audit and checkpoint sidecar do not agree on the number of deduplicated images; the notebook reports both instead of smoothing over the discrepancy. The dataset's image rights are unresolved, and nothing here implies legal clearance.

**Suggested presentation flow:** dataset → preprocessing → architecture → training evidence → held-out metrics → live prediction.

## Contents

1. [Project introduction](#1-project-introduction)
2. [Libraries and dependencies](#2-libraries-and-dependencies)
3. [Dataset description](#3-dataset-description)
4. [Image preprocessing](#4-image-preprocessing)
5. [Model architecture](#5-model-architecture)
6. [Training process](#6-training-process)
7. [Model evaluation](#7-model-evaluation)
8. [Training visualizations](#8-training-visualizations)
9. [Live prediction demonstration](#9-live-prediction-demonstration)
10. [Conclusion and project-file evidence](#10-conclusion-and-project-file-evidence)

## 1. Project Introduction

**NutriCare** is a nutrition-tracking application. Its food-recognition feature accepts a food photograph and predicts one dominant food category. Recognition can reduce the amount of manual food entry and provide a useful starting label for a meal log.

The image classifier is only one part of nutrition tracking: a predicted name does not reveal portion weight, recipe, ingredients, cooking oil, or allergens. Nutrition estimation is handled separately by NutriCare's catalogue and serving-size workflow. Predictions are unconfirmed, and a classifier confidence score is not a calibrated probability of correctness.

## 2. Libraries and Dependencies

The notebook uses PyTorch and Torchvision for the shipped network and its transforms, Pillow for image I/O, NumPy for confusion-matrix plotting, Pandas for readable tables, and Matplotlib for charts. It reuses `nutricare.vision.Predictor` for actual inference instead of creating a second model or preprocessing path. Scikit-learn is not needed because the committed sidecar already contains measured evaluation metrics.

From the repository root, install the ML and notebook packages if needed:

```bash
python -m pip install -r requirements-training.txt
python -m pip install notebook ipykernel pandas matplotlib ipywidgets
```

The full training dependency file is used here because it installs the same Torch/Torchvision/Pillow stack expected by the model. The notebook does **not** start training.

In [ ]:
# Imports and repository discovery (works when started from the repository root or notebooks/).
from pathlib import Path
import hashlib
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from PIL import Image
from IPython.display import display

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (_here, *_here.parents)
     if (candidate / "nutricare" / "vision.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the NutriCare repository. Start Jupyter from the "
        "repository root (or its notebooks/ directory)."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from nutricare import vision  # the application's shared inference implementation

CHECKPOINT_PATH = PROJECT_ROOT / "models" / "v2" / "food_classifier_indian_80_224.pt"
METRICS_PATH = CHECKPOINT_PATH.with_suffix(".metrics.json")
AUDIT_PATH = PROJECT_ROOT / "data" / "v2" / "indian_audit.json"

for required in (CHECKPOINT_PATH, METRICS_PATH, AUDIT_PATH):
    if not required.is_file():
        raise FileNotFoundError(f"Required NutriCare artifact is missing: {required.relative_to(PROJECT_ROOT)}")

with METRICS_PATH.open(encoding="utf-8") as f:
    metrics = json.load(f)
with AUDIT_PATH.open(encoding="utf-8") as f:
    dataset_audit = json.load(f)

print(f"Project: {PROJECT_ROOT}")
print(f"PyTorch {torch.__version__} | Torchvision {torchvision.__version__} | NumPy {np.__version__}")
print(f"Primary checkpoint present: {CHECKPOINT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Metrics sidecar present: {METRICS_PATH.relative_to(PROJECT_ROOT)}")

## 3. Dataset Description

### Source lineage and licensing

The primary checkpoint's metrics sidecar identifies a **3,997-image snapshot** from `SoumyaXD/Indian-Food-Classifier`. Repository forensic notes identify those files as byte-identical to a 3,997-image subset of the 4,000 images in `gnpaone/Indian-food-classification`; that repository links the corpus to Kaggle's `iamsouravbanerjee/indian-food-images-dataset`.

These are lineage statements, not proof of permission. The upstream image terms and chain of title were not verified; the mirror repositories do not grant rights to third-party photographs. The model card consequently classifies the primary dataset/checkpoint as **UNRESOLVED — internal experimental use only; do not redistribute**. Do not describe this set as legally cleared.

### Counts: do not combine unlike measurements

The tracked audit reports 4,000 raw files and 3,474 remaining after an exact-MD5 plus near-duplicate dHash audit. Separately, the **model sidecar says it trained on 3,997 images** and records splits totalling 3,997 (2,705 / 645 / 647). The audit's 3,474 deduplicated figure and the checkpoint's 3,997-image accounting are not reconciled by the repository evidence. The notebook keeps each as a distinct reported count; it does not assert that 3,474 is the model's training count or that the model used the audit's deduplication output.

The training split distribution is nearly uniform in the sidecar: 50 images per class except `cham_cham` (47). The held-out test support is 647 images. Split policy in the sidecar is described as duplicate-group-aware, per-class stratified 70/15/15.

In [ ]:
# Build the class table using the model-sidecar class list as the authoritative output order.
class_names = metrics["class_names"]
raw_counts = dataset_audit.get("per_class_raw", {})
audit_dedup_counts = dataset_audit.get("per_class_after_dedupe", {})
model_counts = metrics["dataset"].get("per_class", {})
per_class_test = metrics.get("per_class", {})

class_table = pd.DataFrame({
    "Food class (checkpoint output order)": class_names,
    "Raw images (audit)": [raw_counts.get(name, np.nan) for name in class_names],
    "After audit dedupe (audit)": [audit_dedup_counts.get(name, np.nan) for name in class_names],
    "Images reported in model dataset": [model_counts.get(name, np.nan) for name in class_names],
    "Test support": [per_class_test.get(name, {}).get("support", np.nan) for name in class_names],
})

raw_total = int(dataset_audit.get("images_total", sum(raw_counts.values())))
dedup_total = int(dataset_audit.get("images_after_dedupe", sum(audit_dedup_counts.values())))
model_total = int(metrics["dataset"]["total_images"])
split_sizes = metrics["dataset"]["split_sizes"]
print(f"Raw audit files: {raw_total:,} | audit after deduplication: {dedup_total:,}")
print(f"Model-sidecar dataset: {model_total:,} | split totals: {sum(split_sizes.values()):,} "
      f"({split_sizes['train']:,} train / {split_sizes['val']:,} validation / {split_sizes['test']:,} test)")
print("The 3,474 vs 3,997 count discrepancy is unresolved in the committed evidence.")
display(class_table)

In [ ]:
# A class distribution chart from the checkpoint's reported training-image counts.
plot_table = class_table.sort_values("Images reported in model dataset", ascending=True)
fig, ax = plt.subplots(figsize=(12, 18))
ax.barh(plot_table["Food class (checkpoint output order)"],
        plot_table["Images reported in model dataset"], color="#168C83")
ax.set_title("Training image counts by class (checkpoint sidecar)")
ax.set_xlabel("Images reported in model's dataset")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

### Category names and sample images

The model predicts the 80 labels stored in checkpoint output order (listed in the table above). This order is the model's mapping; it is not the order of NutriCare's broader food catalogue.

The large raw image archive is intentionally not committed to the repository. If it is available locally, set `NUTRICARE_INDIAN_FOOD_DATASET` to the folder containing one subfolder per class. The code below looks for real files in that folder and a few common local dataset locations. It does not substitute unrelated site assets or fabricate example food images. With a clean clone and no local dataset archive, sample-image display is skipped and reported.

In [ ]:
# Find a local class-folder dataset if the user has one. The archive is not bundled in this repo.
_candidate_roots = []
if os.environ.get("NUTRICARE_INDIAN_FOOD_DATASET"):
    _candidate_roots.append(Path(os.environ["NUTRICARE_INDIAN_FOOD_DATASET"]).expanduser())
_candidate_roots.extend([
    PROJECT_ROOT / "data" / "raw" / "Indian-Food-Images",
    PROJECT_ROOT / "data" / "raw" / "indian_food_80",
    PROJECT_ROOT / "data" / "indian_food_80",
    PROJECT_ROOT / "data" / "v2" / "images",
])
IMAGE_DATASET_ROOT = next(
    (root.resolve() for root in _candidate_roots
     if root.is_dir() and any((root / name).is_dir() for name in class_names)),
    None,
)

if IMAGE_DATASET_ROOT is None:
    print("No local source-dataset image folder found. Set NUTRICARE_INDIAN_FOOD_DATASET "
          "to a folder with class-name subfolders to show source samples.")
else:
    print(f"Using local image folder: {IMAGE_DATASET_ROOT}")
    examples = []
    for slug in class_names:
        class_dir = IMAGE_DATASET_ROOT / slug
        if class_dir.is_dir():
            examples.extend(sorted(p for p in class_dir.iterdir()
                                   if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})[:1])
        if len(examples) >= 8:
            break
    if not examples:
        print("The detected root had no directly readable class examples.")
    else:
        cols = 4
        rows = int(np.ceil(len(examples) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(14, 3.8 * rows), squeeze=False)
        for ax, path in zip(axes.flat, examples):
            with Image.open(path) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(path.parent.name)
            ax.axis("off")
        for ax in axes.flat[len(examples):]:
            ax.axis("off")
        fig.suptitle("Real local dataset samples (source images; not bundled)", y=1.01)
        plt.tight_layout()
        plt.show()

## 4. Image Preprocessing

NutriCare's existing inference code loads the checkpoint's input size and normalization values, converts the image to RGB, then applies the checkpoint's evaluation transform: resize the shorter side to `int(input_size × 256 / 224)`, center-crop to the model input size, convert to a tensor, and normalize each RGB channel. For the shipped 224-pixel checkpoint, this is a resize to 256 followed by a 224 × 224 center crop.

The classifier is used in evaluation mode; **no random augmentation is applied at inference**. Training-time augmentation settings for this exact historical checkpoint are not fully recorded in its sidecar, so the notebook does not present augmentation settings from a different trainer as if they were verified training settings.

The before/after display below uses the model's own predictor transform, not a notebook-specific approximation.

In [ ]:
# Initialize the same shared predictor used by the application, pointing at its primary checkpoint.
predictor = vision.Predictor(model_path=CHECKPOINT_PATH)
predictor._load()  # load once to expose the actual model transform for this demonstration

# Get normalization values from the saved checkpoint metadata for a faithful display-only inverse.
checkpoint_payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
normalise_mean = checkpoint_payload.get("normalise_mean", [0.485, 0.456, 0.406])
normalise_std = checkpoint_payload.get("normalise_std", [0.229, 0.224, 0.225])
input_size = int(checkpoint_payload.get("input_size", 224))
print(f"Application transform: {predictor._transform}")
print(f"Checkpoint input: {input_size} × {input_size}; RGB mean={normalise_mean}; std={normalise_std}")

In [ ]:
# Display a real before/after only if a dataset image is locally available.
if IMAGE_DATASET_ROOT is None or not examples:
    print("Before/after image example skipped: no local source image bytes are present.")
else:
    sample_path = examples[0]
    with Image.open(sample_path) as opened:
        original = opened.convert("RGB")
    transformed = predictor._transform(original)  # exactly the inference transform
    mean_t = torch.tensor(normalise_mean).view(3, 1, 1)
    std_t = torch.tensor(normalise_std).view(3, 1, 1)
    display_tensor = (transformed.cpu() * std_t + mean_t).clamp(0, 1)
    transformed_rgb = display_tensor.permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(original)
    axes[0].set_title(f"Original • {original.width} × {original.height}")
    axes[1].imshow(transformed_rgb)
    axes[1].set_title(f"Inference input • {input_size} × {input_size}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 5. Model Architecture

The deployed primary classifier is **MobileNetV3 Small** with a 224 × 224 RGB input and 80 output scores. MobileNetV3 is a convolutional neural network designed for efficient inference, making the small variant a practical choice for a responsive food-scanning feature. Its output layer was adapted to the 80 food labels.

The model card and metrics metadata say this checkpoint was **warm-started from NutriCare's 12-class MobileNetV3 Small checkpoint, with the classification head reinitialized** for the 80 food classes. This is a transfer-learning-style initialization from an existing food model. It is not accurate to claim that this 80-class checkpoint's head or full network was loaded directly from ImageNet. The existing vision module's `build_model` function constructs the same model family and classifier head used by runtime inference.

In [ ]:
# A compact, readable summary from the actual loaded checkpoint model.
loaded_model = predictor._model
parameter_count = sum(parameter.numel() for parameter in loaded_model.parameters())
trainable_count = sum(parameter.numel() for parameter in loaded_model.parameters() if parameter.requires_grad)

print(f"Runtime model class: {type(loaded_model).__name__}")
print(f"Feature extractor stages: {len(loaded_model.features)}")
print(f"Classifier head:\n{loaded_model.classifier}")
print(f"Parameters: {parameter_count:,} total ({trainable_count:,} marked trainable in eval instance)")
print(f"Output classes: {len(predictor._classes)}")
assert len(predictor._classes) == len(class_names) == 80, "Checkpoint output mapping differs from its metrics sidecar."

In [ ]:
# Display the exact model output index -> label mapping used by this loaded predictor.
mapping_table = pd.DataFrame([
    {"Output index": i, "Checkpoint class": class_names[i],
     "Application display label": predictor._classes[i].get("label", predictor._classes[i]["slug"])}
    for i in range(len(class_names))
])
display(mapping_table)

## 6. Training Process

The committed artifact records a **25-epoch CPU training run** (about 3,027.5 seconds in its metadata) on a 70/15/15 train/validation/test split. The saved history marks epochs 1–3 as the **head** phase and epochs 4–25 as **fine-tuning**. The sidecar records per-epoch validation metrics and separately stores held-out test metrics. It does not preserve enough training metadata to verify the checkpoint-selection rule.

### What the committed evidence does and does not specify

| Setting | Evidence for this shipped 80-class checkpoint |
|---|---|
| Architecture / input | MobileNetV3 Small / 224 px |
| Initialization | Warm-started from NutriCare 12-class MobileNetV3 Small; classification head reset |
| Epochs / device | 25 / CPU |
| Split | 2,705 train / 645 validation / 647 test; sidecar says duplicate-group-aware, stratified 70/15/15 |
| Training phases | Epochs 1–3 `head`; epochs 4–25 `finetune` (from saved history) |
| Batch size, optimizer, learning rate, loss function | **Not recorded in the saved 80-class metrics sidecar / model card** |
| Exact augmentation recipe and epoch-level training loss | **Not fully available for this artifact** |

There are other training scripts in this repository, but they define separate training paths/checkpoints. Their command defaults are not evidence for the historical primary artifact's missing hyperparameters, so this presentation does not substitute those settings. We use the genuine saved epoch history below and do not launch any training run.

## 7. Model Evaluation

The values below are the committed **held-out test split** results for the 80-class primary model (647 test images). Accuracy is the fraction of test predictions exactly correct. Precision reflects how often predictions for a class were correct; recall reflects how many actual examples of that class were found; F1 combines precision and recall. **Macro F1** gives every class equal weight, while weighted F1 weights classes by test support. The raw softmax confidence is not calibrated.

The test result is not phone-camera field accuracy, a nutrition-safety validation, or a guarantee of a correct result for a particular photo.

In [ ]:
# Metrics are loaded from the model's committed sidecar, not recomputed or invented.
results = metrics["results"]
metric_rows = [
    ("Accuracy", results["test_accuracy"], "Exact correctness on the held-out test split"),
    ("Macro precision", results["macro_precision"], "Mean precision across the 80 classes"),
    ("Macro recall", results["macro_recall"], "Mean recall across the 80 classes"),
    ("Macro F1", results["macro_f1"], "Equal weight per class"),
    ("Weighted F1", results["weighted_f1"], "F1 weighted by test support"),
    ("Test loss", results["test_loss"], "Sidecar's held-out test loss"),
]
metric_table = pd.DataFrame(metric_rows, columns=["Metric", "Measured value", "Interpretation"])
metric_table["Measured value"] = metric_table["Measured value"].map(lambda value: f"{value:.4f}")
display(metric_table)
print(f"Split: TEST | n={results['test_support']} | "
      f"train accuracy reported separately in sidecar: {results['train_accuracy']:.4f}")
print("The train accuracy is not a test score. The sidecar reports a "
      f"train–test accuracy gap of {results['train_test_gap']:.4f}.")

In [ ]:
# Per-class test metrics and supports, from the saved evaluation report.
per_class_table = pd.DataFrame.from_dict(metrics["per_class"], orient="index")
per_class_table.index.name = "Class"
per_class_table = per_class_table.rename(columns={
    "precision": "Precision", "recall": "Recall", "f1": "F1", "support": "Test support"
}).sort_values("F1")
display(per_class_table.style.format({"Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}"}))
print("Lowest-F1 classes in the recorded test evaluation:",
      ", ".join(f"{name} ({row['F1']:.3f})" for name, row in per_class_table.head(6).iterrows()))

In [ ]:
# Confusion matrix: rows are true classes; columns are predicted classes.
confusion = np.asarray(metrics["confusion_matrix"], dtype=int)
labels = metrics["class_names"]
if confusion.shape != (len(labels), len(labels)):
    raise ValueError(f"Unexpected confusion matrix shape: {confusion.shape}")
fig, ax = plt.subplots(figsize=(18, 16))
image = ax.imshow(confusion, interpolation="nearest", cmap="Blues")
ax.set_title("80-class held-out TEST confusion matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=6)
ax.set_yticklabels(labels, fontsize=6)
fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02, label="Number of test images")
plt.tight_layout()
plt.show()

## 8. Training Visualizations

These plots use the checkpoint's **actual saved epoch history**. It contains epoch-level training accuracy and validation accuracy. Validation loss is present only for epochs where recorded in the sidecar; training loss is not included for this run, so the notebook does not invent or reconstruct a training-loss curve. Gaps in a plotted series remain gaps.

In [ ]:
history = pd.DataFrame(metrics.get("history", []))
if history.empty:
    print("No historical epoch log is present in the model sidecar; no training plots can be drawn.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    if "train_acc" in history and "val_acc" in history:
        axes[0].plot(history["epoch"], history["train_acc"], marker="o", ms=3, label="Training accuracy")
        axes[0].plot(history["epoch"], history["val_acc"], marker="o", ms=3, label="Validation accuracy")
        axes[0].set_title("Recorded training and validation accuracy")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Accuracy")
        axes[0].legend()
        axes[0].grid(alpha=0.25)
    else:
        axes[0].text(0.5, 0.5, "Accuracy history unavailable", ha="center", va="center")
        axes[0].set_axis_off()

    if "val_loss" in history and history["val_loss"].notna().any():
        axes[1].plot(history["epoch"], history["val_loss"], marker="o", ms=3,
                     color="#C46A27", label="Validation loss")
        axes[1].set_title("Recorded validation loss (where present)")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].legend()
        axes[1].grid(alpha=0.25)
        if "phase" in history:
            axes[1].text(0.02, 0.98,
                         "Training loss was not saved in this sidecar.",
                         transform=axes[1].transAxes, va="top", fontsize=9,
                         bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"})
    else:
        axes[1].text(0.5, 0.5, "Validation loss history unavailable", ha="center", va="center")
        axes[1].set_axis_off()
    plt.tight_layout()
    plt.show()

    display(history)

## 9. Live Prediction Demonstration

The cell below accepts an image through an optional Jupyter upload control. If widgets are unavailable, set `USER_IMAGE_PATH` to a local image path and run the prediction cell. Inference calls the application's `vision.Predictor` with the primary 80-class checkpoint, so the label mapping, image validation, preprocessing, network, and softmax scores are the same runtime path NutriCare uses.

The output is a **single-label** top prediction and optional ranked alternatives. NutriCare withholds the food label when the top raw softmax score is **below 0.35**, returning “food not recognized” instead. This is an abstention cutoff, not an invalid-image result or calibrated probability of correctness. A score at or above the cutoff still does not confirm the identification.

In [ ]:
# Optional image upload. If ipywidgets is unavailable, use USER_IMAGE_PATH below instead.
USER_IMAGE_PATH = None  # Example: Path("my_food_photo.jpg")
upload_widget = None
try:
    import ipywidgets as widgets
    from IPython.display import display
    upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="Choose food image")
    display(upload_widget)
except ImportError:
    print("ipywidgets is not installed. Set USER_IMAGE_PATH to a local image file to continue.")

In [ ]:
def _uploaded_file_bytes(widget):
    """Handle both ipywidgets 7 (dict) and 8 (tuple of uploaded-file dicts)."""
    if widget is None or not widget.value:
        return None, None
    value = widget.value
    if isinstance(value, dict):
        name, details = next(iter(value.items()))
    else:
        details = value[0]
        name = details.get("name", "uploaded-image")
    return details.get("content"), name

image_bytes, image_name = _uploaded_file_bytes(upload_widget)
if image_bytes is not None:
    image_bytes = bytes(image_bytes)
if image_bytes is None and USER_IMAGE_PATH:
    image_path = Path(USER_IMAGE_PATH).expanduser()
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")
    image_bytes = image_path.read_bytes()
    image_name = image_path.name

if image_bytes is None:
    print("No image selected yet. Upload one above or set USER_IMAGE_PATH, then rerun this cell.")
else:
    try:
        with Image.open(__import__("io").BytesIO(image_bytes)) as uploaded:
            preview = uploaded.convert("RGB")
        display(preview)
        prediction = predictor.predict(image_bytes, top_k=5)
        print(f"Image: {image_name} | dimensions: {prediction['image']['width']} × {prediction['image']['height']}")
        threshold = vision.MIN_FOOD_RECOGNITION_CONFIDENCE
        score = prediction["top"]["confidence"]
        if score < threshold:
            print("Food not recognized: the top raw softmax score is below NutriCare's 35% minimum.")
            print(f"Top score: {score * 100:.1f}% | minimum: {threshold * 100:.0f}%")
            print("Try a clear photo of one dish, or choose a food manually.")
        else:
            print(f"Predicted food: {prediction['top']['label']} ({prediction['top']['slug']})")
            print(f"Raw softmax confidence: {prediction['top']['confidence_pct']:.1f}%")
            print("Confidence is not calibrated; prediction remains unconfirmed.")
            display(pd.DataFrame(prediction["predictions"])[["label", "slug", "confidence_pct"]])
    except (vision.ImageError, vision.ModelUnavailable) as exc:
        print(f"NutriCare prediction could not be completed: {exc}")

## 10. Conclusion and Project-File Evidence

### Summary

- **Dataset:** 80 Indian-food categories, with a 4,000-image raw audit. The model sidecar reports 3,997 images and a 2,705 / 645 / 647 split. A separate whole-dataset deduplication audit reports 3,474 remaining images; the difference from the model's reported 3,997 is unresolved.
- **Model:** MobileNetV3 Small, 224-pixel input, 80 outputs, warm-started from NutriCare's 12-class model with the output head reset.
- **Training:** 25 CPU epochs; first three recorded as head training, then fine-tuning. Batch size, exact optimizer, learning rate, loss, and some augmentation details are not available in the saved evidence. The notebook does not launch training.
- **Held-out evaluation:** test accuracy **0.5672**, macro precision **0.5776**, macro recall **0.5656**, macro F1 **0.5559**, weighted F1 **0.5575** on **647** test images. The reported training accuracy is 0.9656; it is not a test metric, and the large train–test gap is a warning sign.
- **Recognition behavior and limitations:** the scanner abstains below 0.35 raw softmax confidence; this cutoff is not calibrated. Image rights remain unresolved; the model predicts one dominant class per image; performance on naturally captured user photos is not established; food labels do not measure portions or nutrition.

### Actual project files used

| Project file | Why it is used |
|---|---|
| `models/v2/food_classifier_indian_80_224.pt` | Current primary scanner checkpoint |
| `models/v2/food_classifier_indian_80_224.metrics.json` | Dataset counts, stored class order, epoch history, test results and per-class report |
| `data/v2/indian_audit.json` | Raw and post-deduplication image audit (separately reported from model-sidecar counts) |
| `nutricare/vision.py` | Application model loader, checkpoint architecture builder, transforms, class mapping and inference |
| `data/food_classes.json` | Application display-label taxonomy used by inference when a slug maps to it |
| `docs/MODEL.md` | Current primary/fallback policy, provenance and model limitations |
| `docs/DATASET.md` | Source dataset context and acquisition notes |
| `docs/DATASET_AUDIT_V2.md` and `docs/LICENSING_AUDIT.md` | Audit, contamination and rights caveats |
| `training/train_food_classifier.py`, `training/train_v2.py` | Repository training code reviewed; their configs are not substituted for missing historical primary-checkpoint hyperparameters |

### Open and run locally

1. Open a terminal in the NutriCare repository root.
2. Install dependencies with the commands in Section 2.
3. Start Jupyter with `jupyter notebook` (or open this `.ipynb` in VS Code with its Jupyter extension).
4. Open `notebooks/NutriCare_Food_Recognition.ipynb` and choose **Run All**.
5. To display training-source examples, separately obtain the dataset in accordance with its rights and set `NUTRICARE_INDIAN_FOOD_DATASET` to its class-folder root. To run live prediction, upload an image in Section 9 or set `USER_IMAGE_PATH`.

No source image archive is shipped in this repository, so sample-image and before/after cells skip gracefully unless a local copy is supplied. The checkpoint, sidecar, evaluation tables/charts, and prediction from a user-supplied image are otherwise demonstrated from actual project artifacts. The notebook is presentation/review tooling only; it does not modify application behavior or model files.